In [1]:
# useful libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

print(f'XGBoost version: {xgb.__version__}')

XGBoost version: 3.2.0


In [2]:
# setting up default plotting parameters
%matplotlib inline

plt.rcParams['figure.figsize'] = [20.0, 7.0]
plt.rcParams.update({'font.size': 22,})

sns.set_palette('viridis')
sns.set_style('white')
sns.set_context('talk', font_scale=0.8)

In [3]:
# read in data
df = pd.read_csv('train.csv')

print(df.shape)
df.head()

(967332, 14)


,auctionId,timeStamp,placementId,websiteId,hashedRefererDeepThree,country,opeartingSystem,browser,browserVersion,device,environmentType,integrationType,articleSafenessCategorization,isSold
0,001ed16b-dd08-4599-b8ef-4f56a373c454_6e5f1087-...,1603815466,120706,68203,1ae7c2d3c28b711c072d8e2eb3869fa59090669bdc153e...,US,Windows,Chrome,86_0,PC,js-web,2,safe,False
1,0024b36a-4fb5-4070-88fb-fc0bfb1909ed,1603974586,69454,42543,df1108bf6ae49dbccf5eab60ff9d04a6a09dda60ec7290...,RO,Android,Facebook App,293_0,Phone,js-fbwv,1,unsafe,False
2,003630fa-ad63-4283-be1b-141670132d70_f37c2b23-...,1604229969,100170,57703,cc6957e8aec85a4d920991c53874c5d0780bbfbd469802...,UK,Android,Facebook App,294_0,Phone,js-web,2,safe,True
3,0048c65a-ce76-43ba-98d2-8e87607468f8,1604156610,100446,57797,7fc0bb7a65d074e003cce786cda2b070f80dd47179c4b9...,ES,Android,Chrome Mobile,86_0,Phone,js-ampsf,1,safe,True
4,0056b8a7-54f9-4ac8-8d50-f725bf377872,1604004493,119517,67613,3a6552ccbf66ad166aa9005c3e08f70716abd676cfd87b...,FR,Android,Facebook App,293_0,Phone,js-fbwv,1,unsafe,False


In [4]:
print(df.isSold.value_counts())

df = df.rename(columns={'opeartingSystem': 'operatingSystem'})

isSold
False    533425
True     433907
Name: count, dtype: int64


In [5]:
df.set_index('auctionId', inplace = True, drop = True)

In [6]:
# Data preprocessing

X = df.copy()

class_dummies = pd.get_dummies(X['country'], prefix = 'country')
X = X.join(class_dummies)

class_dummies = pd.get_dummies(X['operatingSystem'], prefix = 'operatingSystem')
X = X.join(class_dummies)

class_dummies = pd.get_dummies(X['browser'], prefix = 'browser')
X = X.join(class_dummies)

class_dummies = pd.get_dummies(X['device'], prefix = 'device')
X = X.join(class_dummies)

class_dummies = pd.get_dummies(X['environmentType'], prefix = 'environmentType')
X = X.join(class_dummies)

class_dummies = pd.get_dummies(X['articleSafenessCategorization'], prefix = 'articleSafenessCategorization')
X = X.join(class_dummies)

X = X.drop(columns=['country', 'operatingSystem', 'browser', 'device', 'environmentType', 'articleSafenessCategorization', 'hashedRefererDeepThree', 'browserVersion', 'isSold'])

In [7]:
from sklearn.model_selection import train_test_split

y = df['isSold'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from itertools import product
from tqdm.notebook import tqdm
import time

# 1. Grille d'hyperparamètres
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

# 2. Ratio de classes
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos = n_neg / n_pos

# 3. Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X_train, y_train))

# 4. Toutes les combinaisons
param_names = list(param_grid.keys())
all_combos = [dict(zip(param_names, v)) for v in product(*param_grid.values())]

best_score = -1
best_params = None

# 5. Boucle avec barre de progression
pbar = tqdm(all_combos, desc='Grid Search GPU', unit='combo')

for params in pbar:
    fold_scores = []
    for train_idx, val_idx in folds:
        model = xgb.XGBClassifier(
            **params, device='cuda', scale_pos_weight=scale_pos,
            random_state=42, eval_metric='logloss', verbosity=0,
        )
        model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        y_pred_fold = model.predict(X_train.iloc[val_idx])
        fold_scores.append(f1_score(y_train.iloc[val_idx], y_pred_fold))

    mean_f1 = np.mean(fold_scores)
    if mean_f1 > best_score:
        best_score = mean_f1
        best_params = params

    pbar.set_postfix({'best_F1': f'{best_score:.4f}', 'cur_F1': f'{mean_f1:.4f}'})

print(f'\n✅ Meilleurs params : {best_params}')
print(f'Meilleur F1 CV : {best_score:.4f}')

# 6. Ré-entraîner le meilleur modèle
best_xgb = xgb.XGBClassifier(
    **best_params, device='cuda', scale_pos_weight=scale_pos,
    random_state=42, eval_metric='logloss', verbosity=0,
)
best_xgb.fit(X_train, y_train)


Ratio classes (neg/pos): 1.23
Lancement de l'optimisation des hyperparamètres sur GPU...
Fitting 5 folds for each of 144 candidates, totalling 720 fits


Exception ignored while calling ctypes callback function <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x000002D41D0E8940>>:
Traceback (most recent call last):
  File "C:\Users\morot\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\core.py", line 607, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument
KeyboardInterrupt: 


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, recall_score, precision_score

y_pred = best_xgb.predict(X_test)

print('--- Performances sur le jeu de Test ---')
print('Accuracy :        ', accuracy_score(y_test, y_pred))
print('Precision score : ', precision_score(y_test, y_pred))
print('F1 score :        ', f1_score(y_test, y_pred))
print('Recall score :    ', recall_score(y_test, y_pred))

print('\nMatrice de confusion :')
print(confusion_matrix(y_test, y_pred))